## 1. Environment & Package Imports

- Downloads the Food-11 zipped dataset via Google Drive link using the gdown utility and extracts the image directory structures to the current workspace.


In [ ]:
# Google Drive
!gdown --id '149paISvxCXDlr-720UEzseQA-y53rZxN' --output food-11.zip

# Dropbox
# !wget "https://www.dropbox.com/s/7yl5rra84ia0k8f/food-11.zip?dl=0" -O food-11.zip

# MEGA
# !wget https://megatools.megous.com/builds/megatools-1.10.3.tar.gz
# !tar -zxvf /content/megatools-1.10.3.tar.gz
# !sudo apt-get install libtool libglib2.0-dev gobject-introspection libgmp3-dev nettle-dev asciidoc glib-networking openssl libcurl4-openssl-dev libssl-dev
# %cd megatools-1.10.3/
# !./configure make
# !sudo make install
# %cd /content/
# !megadl 'https://mega.nz/file/FdlygByK#QQ5LP71MMjeXrXwoXM2qlygUsJ1D-6d5fMhj5gyi2Vc'

!unzip food-11.zip

## 2. Import Packages

- Imports core PyTorch libraries, neural network layers, torchvision transformers for data augmentation, dataset packaging utilities, and tqdm progress bars.


In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import numpy as np
from PIL import Image

# Data grouping utilities tailored for semi-supervised training pipelines
from torch.utils.data import ConcatDataset, DataLoader, Subset, TensorDataset
from torchvision.datasets import DatasetFolder

# Interactive progress bars wrapper
from tqdm import tqdm

## 3. Image Augmentation Configurations (Transforms)

- Establishes strong stochastic transformations (rotation, crop, color jitter) for training to alleviate deep network overfitting, while confining testing streams to clean fixed resizing filters.


In [ ]:
# not every augmentation is useful.
# Please think about what kind of augmentation is helpful for food recognition.
# Strong data augmentation recipe engineered specifically to boost food image classification
train_tfm = transforms.Compose(
    [
        transforms.Resize((128, 128)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
        transforms.RandomResizedCrop(128, scale=(0.8, 1.0)),
        transforms.ToTensor(),  # Standardize raw pixel grids to floating-point tensors scaled [0.0, 1.0]
    ]
)

# We don't need augmentations in testing and validation.
# All we need here is to resize the PIL image and transform it into Tensor.
# Evaluation transformations must remain completely deterministic without stochastic variance
test_tfm = transforms.Compose(
    [
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
    ]
)

## 4. Dual-View Dataset & DataLoader Formulations

- Sets up folder pipelines, initializing parallel multi-threaded data loading streams and mapping out a twin-view unlabeled pipeline (unlabeled_set and unlabeled_set_pseudo) to prepare for semi-supervised modeling.


In [ ]:
batch_size = 128

# Ingest labeled training image folder paths using strong training transforms
train_set = DatasetFolder(
    "food-11/training/labeled",
    loader=lambda x: Image.open(x),
    extensions="jpg",
    transform=train_tfm,
)

# Ingest baseline validation images using clean evaluation test transforms
valid_set = DatasetFolder(
    "food-11/validation",
    loader=lambda x: Image.open(x),
    extensions="jpg",
    transform=test_tfm,
)

# Ingest unlabeled instances wrapped with strong geometric distortion pipelines
unlabeled_set = DatasetFolder(
    "food-11/training/unlabeled",
    loader=lambda x: Image.open(x),
    extensions="jpg",
    transform=train_tfm,
)

# Ingest duplicate unlabeled folders wrapped with clean transforms to infer unbiased predictions
unlabeled_set_pseudo = DatasetFolder(
    "food-11/training/unlabeled",
    loader=lambda x: Image.open(x),
    extensions="jpg",
    transform=test_tfm,  # Clean image version ensuring high confidence label selection
)

test_set = DatasetFolder(
    "food-11/testing",
    loader=lambda x: Image.open(x),
    extensions="jpg",
    transform=test_tfm,
)

# Assemble structural batch loaders allocating multiple worker threads and memory pinning optimization
train_loader = DataLoader(
    train_set, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True
)
valid_loader = DataLoader(
    valid_set, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True
)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

## 5. Model Architecture Declaration (ResNet18 Backbone)

- Reuses the classic ResNet18 backbone topology without pre-trained parameters, dynamically extracting structural link shapes to redefine the fully connected head to match 11 food classes.


In [ ]:
# 5. 建立模型
import torchvision.models as models
import torch.nn as nn


class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        # Instantiate ResNet18 feature extractor architecture from scratch
        self.model = models.resnet18(weights=None)
        # Isolate layer dimension sizes and re-shape the output classification block
        in_features = self.model.fc.in_features
        self.model.fc = nn.Linear(
            in_features, 11
        )  # Map parameters into 11 distinct category outputs

    def forward(self, x):
        # Direct inputs across resnet layer computational pipelines
        return self.model(x)

## 6. Custom Semi-Supervised Pseudo-Labeling Function (v2 Optimized)

- Evaluates unlabeled data in inference mode, filtering out highly confident samples exceeding a target threshold (0.85). It extracts the indices and corresponding detached labels to prevent memory leaks.


In [ ]:
# # 6. 半監督學習 (Pseudo-labeling)
# def get_pseudo_labels(dataset, model, threshold=0.85):
#     device = "cuda" if torch.cuda.is_available() else "cpu"
#     data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
#     model.eval()
#     softmax = nn.Softmax(dim=-1)

#     # 儲存符合條件的影像與偽標籤
#     pseudo_indices_list, pseudo_labels_list = [], []
#     sample_idx = 0

#     for batch in tqdm(data_loader, desc="Pseudo-labeling"):
#         imgs, _ = batch
#         with torch.no_grad():
#             logits = model(imgs.to(device))
#         probs = softmax(logits)
#         max_probs, pred_labels = torch.max(probs, dim=1)

#         # 過濾高信心樣本
#         mask = max_probs > threshold
#         if mask.any():
#             batch_size_actual = len(imgs)
#             batch_indices = torch.arange(sample_idx, sample_idx + batch_size_actual, device="cpu")[mask.cpu()]
#             pseudo_indices_list.append(batch_indices)
#             pseudo_labels_list.append(pred_labels[mask].cpu())

#         sample_idx += len(imgs)

#     model.train()

#     # 回傳符合高信心門檻的偽標籤資料集
#     if len(pseudo_indices_list) == 0:
#         print("No pseudo-labels generated (all below threshold).")
#         return [], torch.tensor([])

#     pseudo_indices = torch.cat(pseudo_indices_list, dim=0).tolist()
#     pseudo_labels = torch.cat(pseudo_labels_list, dim=0)
#     return pseudo_indices, pseudo_labels

In [ ]:
# get_pseudo_labels_v2（回傳索引 + 標籤，支援強增強）
def get_pseudo_labels_v2(dataset, model, threshold=0.85):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    data_loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=False, num_workers=2
    )

    # Toggle model to evaluation mode to ensure accurate confidence scores
    model.eval()
    softmax = nn.Softmax(dim=-1)
    pseudo_indices_list, pseudo_labels_list = [], []
    sample_idx = 0  # Tracks the global absolute index inside the original dataset

    for batch in tqdm(data_loader, desc="Pseudo-labeling"):
        imgs, _ = batch
        with (
            torch.no_grad()
        ):  # Shut down gradient history graph to save memory overhead
            logits = model(imgs.to(device))
        probs = softmax(logits)
        max_probs, pred_labels = torch.max(probs, dim=1)

        # Filter samples that exceed the confident prediction boundary
        mask = max_probs > threshold
        if mask.any():
            batch_size_actual = len(imgs)
            batch_indices = torch.arange(
                sample_idx, sample_idx + batch_size_actual, device="cpu"
            )[mask.cpu()]
            pseudo_indices_list.append(batch_indices)
            # Detach labels from the PyTorch computation graph to avoid memory leaks
            pseudo_labels_list.append(pred_labels[mask].detach().cpu())

        sample_idx += len(imgs)
    model.train()

    if len(pseudo_indices_list) == 0:
        print("No pseudo-labels generated (all below threshold).")
        return [], torch.tensor([])

    # Isolate final list elements and stack tracking arrays together
    pseudo_indices = torch.cat(pseudo_indices_list, dim=0).tolist()
    pseudo_labels = torch.cat(pseudo_labels_list, dim=0)
    return pseudo_indices, pseudo_labels

## 7. Semi-Supervised Training Loop with Dynamic Dataset Concatenation

- Initializes the AdamW optimizer with Label Smoothing (0.1) and a Cosine Annealing scheduler. After a 5-epoch warm-up, it incorporates strongly augmented pseudo-labeled instances into the main training pool.


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = Classifier().to(device)
# Apply Label Smoothing regularizer to prevent network overconfidence
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=80)
# Toggle flag to enable semi-supervised data streaming
do_semi = True


# Custom dataset class to map confidence targets to strong training augmentations
class PseudoDataset(torch.utils.data.Dataset):
    def __init__(self, subset_images, labels):
        self.subset_images = subset_images
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Triggers unlabeled_set's train_tfm to apply random strong augmentations
        img, _ = self.subset_images[idx]
        label = self.labels[idx]
        return img, label


# Main Optimization and Training Loop
for epoch in range(80):
    train_set_to_use = train_set

    # Warm up model on purely labeled resources for 5 epochs before generating pseudo-labels
    if do_semi and epoch >= 5:
        pseudo_indices, pseudo_labels = get_pseudo_labels_v2(
            unlabeled_set_pseudo, model, threshold=0.85
        )
        if len(pseudo_indices) > 0:
            from torch.utils.data import Subset

            # Isolate matching observations tied to strong structural transformation configurations
            strong_pseudo_images_set = Subset(unlabeled_set, pseudo_indices)
            pseudo_set = PseudoDataset(strong_pseudo_images_set, pseudo_labels)
            # Dynamically concatenate the physical training sets together
            train_set_to_use = ConcatDataset([train_set, pseudo_set])

    # Re-instantiate the training batch dataloader flow dynamically
    train_loader = DataLoader(
        train_set_to_use,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
    )

    # Execute batch network updates
    model.train()
    train_loss = 0.0
    train_accs = []

    for batch in tqdm(train_loader, desc=f"Epoch {epoch + 1}"):
        imgs, labels = batch
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss = criterion(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        acc = (logits.argmax(dim=-1) == labels).float().mean()
        train_loss += loss.item()
        train_accs.append(acc)

    train_loss = train_loss / len(train_loader)
    train_acc = sum(train_accs) / len(train_accs)
    scheduler.step()  # Advance learning rate following the cosine annealing trajectory

    # Model Validation Phase
    model.eval()
    valid_loss = 0.0
    valid_accs = []
    with torch.no_grad():
        for batch in valid_loader:
            imgs, labels = batch
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs)
            loss = criterion(logits, labels)
            acc = (logits.argmax(dim=-1) == labels).float().mean()
            valid_loss += loss.item()
            valid_accs.append(acc)
    valid_loss = valid_loss / len(valid_loader)
    valid_acc = sum(valid_accs) / len(valid_accs)

    print(
        f"Epoch {epoch + 1:03d} | Train Acc: {train_acc:.4f} | Valid Acc: {valid_acc:.4f}"
    )

## 8. Testing Set Blended Inference Execution

- Freezes batch normalization and dropout layers using model.eval(). It loops through test batches using placeholder labels and runs forward pass estimations without tracking gradients to optimize computing speed.


In [ ]:
model.eval()
predictions = []

# Process the testing blocks iteratively
for batch in tqdm(test_loader):
    # Note: 'labels' represents arbitrary fake data padding generated by DatasetFolder wrapper
    imgs, labels = batch

    # Turn off gradient calculation engine to speed up execution and conserve memory resources
    with torch.no_grad():
        logits = model(imgs.to(device))

    # Extract index classes mapping out dominant dimension peaks and register entries
    predictions.extend(logits.argmax(dim=-1).cpu().numpy().tolist())

## 9. Production CSV Prediction Document Export

- Opens a streaming writer interface to format and write prediction arrays into a standardized local file (predict.csv) structured for submission.


In [ ]:
with open("predict.csv", "w") as f:
    f.write("Id,Category\n")

    # Iterate through predictions arrays to append localized observations lines
    for i, pred in enumerate(predictions):
        f.write(f"{i},{pred}\n")

# Log out total successful pseudo-label extractions summary info
print(f"Generated {len(pseudo_indices)} pseudo-labeled samples.")